In [27]:
import numpy as np
import cvxpy as cp
import pandas as pd

# Q.1 Solve for optimal Sharpe Ratio portfolio

In [12]:
sigma = np.array([[1, 0.5, 0, 0, 0, 0], 
                  [0.5, 2, 0, 0, 0, 0],
                  [0, 0, 6, 0, 0, 0],
                  [0, 0, 0, 0, 0, 0],
                  [0, 0, 0, 0, 0, 0],
                  [0, 0, 0, 0, 0, 0]
                 ])

A = np.array([[-2, -1, 0, 1, 1, 0],# y1, y2, y3, a, b, k
              [-1, -4, 0, 3, 1, 0],
              [0, 0, -12, 6, 1, 0],
              [1, 3, 6, 0, 0, 0],
              [1, 1, 1, 0, 0, -1]])
b = np.array([0, 0, 0, 1, 0])

B = np.array([[1, 0, 0, 0, 0, 0],# y1, y2, y3, a, b, k
              [0, 1, 0, 0, 0, 0],
              [0, 0, 1, 0, 0, 0],
              [0, 0, 0, 0, 0, 1]])
c = np.array([0, 0, 0, 0])

# Define and solve the CVXPY problem.
y = cp.Variable(6)# y1, y2, y3, a, b, k

prob = cp.Problem(cp.Minimize(cp.quad_form(y, sigma)),
                 [A @ y == b,
                  B @ y >= c])
print('Optimal Sharpe Ratio:', np.sqrt(1/prob.solve()))
print("A solution y is")
print(y.value)

Optimal Sharpe Ratio: 3.251373336211726
A solution y is
[2.70270270e-02 1.35135135e-01 9.45945946e-02 1.89189189e-01
 5.57511951e-17 2.56756757e-01]


In [13]:
allocation = y.value[:3] / y.value[-1]# y / k => x
allocation

array([0.10526316, 0.52631579, 0.36842105])

In [22]:
#Return
return_vec = np.array([[4,6,9]])
optimal_return = (return_vec@allocation)[0]
print('optimal return:', optimal_return)

optimal return: 6.894736842105264


In [23]:
#Variance
cov_mat = np.array([[1, 0.5, 0], 
                    [0.5, 2, 0],
                    [0, 0, 6]])
optimal_variance = allocation.T@cov_mat@allocation
print('optimal variance:', optimal_variance)

optimal variance: 1.4349030470914128


In [25]:
#Optimal Sharpe Ratio
rf = 3
print('Optimal Sharpe Ratio:', (optimal_return - rf)/np.sqrt(optimal_variance))
#Cross check with the optimal Sharpe Ratio obtained during the optimization. (Equal!)

Optimal Sharpe Ratio: 3.2513733362117265


In [26]:
cur_return = 7
(cur_return-rf)/(optimal_return - rf) * (optimal_variance)

1.4736842105263157

# Q.2 Solve for minimal Mean Absolute Variance

In [31]:
return_mat = pd.DataFrame({'year': [i for i in range(1992,2004)],
                           'sp': [7.6, 10.1, 1.3, 37.6, 23, 33.4, 28.6, 21, -9.1, -11.9, -22.1, 28.7],
                           'bond': [7.3, 13.1, -7.3, 25.9, 0.1, 12, 14.5, -7.5, 17.2, 5.5, 15.2, 0.5],
                           'MM': [2.9, 3, 5.5, 5.6, 5.3, 5.5, 4.7, 5.3, 6.4, 1.8, 1.2, 1.0],
                           'nasdaq': [15.5, 14.7, -3.2, 39.9, 22.7, 21.6, 39.6, 85.6, -39.3, -21.1, -31.5, 50]})
assets = ['sp', 'bond', 'MM', 'nasdaq']
return_mat

,year,sp,bond,MM,nasdaq
0,1992,7.6,7.3,2.9,15.5
1,1993,10.1,13.1,3.0,14.7
2,1994,1.3,-7.3,5.5,-3.2
3,1995,37.6,25.9,5.6,39.9
4,1996,23.0,0.1,5.3,22.7
5,1997,33.4,12.0,5.5,21.6
6,1998,28.6,14.5,4.7,39.6
7,1999,21.0,-7.5,5.3,85.6
8,2000,-9.1,17.2,6.4,-39.3
9,2001,-11.9,5.5,1.8,-21.1


In [41]:
mean_vector = return_mat[assets].mean().values
return_diff_mat = return_mat[assets].values - mean_vector
return_diff_mat

array([[ -4.75      ,  -0.74166667,  -1.11666667,  -0.70833333],
       [ -2.25      ,   5.05833333,  -1.01666667,  -1.50833333],
       [-11.05      , -15.34166667,   1.48333333, -19.40833333],
       [ 25.25      ,  17.85833333,   1.58333333,  23.69166667],
       [ 10.65      ,  -7.94166667,   1.28333333,   6.49166667],
       [ 21.05      ,   3.95833333,   1.48333333,   5.39166667],
       [ 16.25      ,   6.45833333,   0.68333333,  23.39166667],
       [  8.65      , -15.54166667,   1.28333333,  69.39166667],
       [-21.45      ,   9.15833333,   2.38333333, -55.50833333],
       [-24.25      ,  -2.54166667,  -2.21666667, -37.30833333],
       [-34.45      ,   7.15833333,  -2.81666667, -47.70833333],
       [ 16.35      ,  -7.54166667,  -3.01666667,  33.79166667]])

## Create the constraint matrix for y_t - z_t = (r_1t - mu_1)x1 + .... + (r_4t - mu_4)x4 for each time step (will be concatenated with the LHS later (part for x1, ..., x4))

In [53]:
slack_val_matrix = np.zeros((len(return_diff_mat), len(return_mat)*2))
running_ind = 0
for i in range(len(return_mat)):
    slack_val_matrix[i, running_ind] = -1.
    slack_val_matrix[i, running_ind + len(return_mat)] = 1.
    running_ind += 1
slack_val_matrix#y1, y2, y3, ...., y12, z1, z2, z3, ...., z12

array([[-1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0., -1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0., -1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0., -1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0., -1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0., -1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0., -1.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  0.,  0.

## Constraint matrices construction => for constraints with equality (A * y = b) and for constraints with inequality (positive_constraint_mat * y >= 0)

In [85]:
slack_val_matrix = np.zeros((len(return_diff_mat), len(return_mat)*2))
running_ind = 0
for i in range(len(return_mat)):
    slack_val_matrix[i, running_ind] = -1.
    slack_val_matrix[i, running_ind + len(return_mat)] = 1.
    running_ind += 1
slack_val_matrix#y1, y2, y3, ...., y12, z1, z2, z3, ...., z12

# Total weight constraint
total_weight_constraint = np.zeros((1,A.shape[1]))
total_weight_constraint[0,:4] = 1# x1 + x2 + x3 + x4 = 1

# Return constraint
return_constraint = np.zeros((1,A.shape[1]))
return_constraint[0,:4] = mean_vector

A = np.concatenate([return_diff_mat, slack_val_matrix], axis=1)
A = np.concatenate([A, total_weight_constraint], axis=0)
b = np.zeros(len(A))
b[len(A)-1] = 1# x1 + x2 + x3 + x4 = 1

objective_coeff = np.ones(A.shape[1])
objective_coeff[:4] = 0#the asset weight is not included in the objective function


positive_constraint_mat = np.eye(A.shape[1])
positive_constraint_mat = np.concatenate([positive_constraint_mat, return_constraint], axis=0)
zero_vector = np.zeros(len(positive_constraint_mat))
zero_vector[len(zero_vector)-1] = 9# Minimum return of 9%

## LP part

In [86]:
# Define and solve the CVXPY problem.
y = cp.Variable(A.shape[1])# x1, x2, x3, x4, y1, y2, ...., y12, z1, z2, ...., z12


prob = cp.Problem(cp.Minimize(objective_coeff @ y),
                              [A @ y == b,
                               positive_constraint_mat @ y >= zero_vector])

print('Optimal MAD:', prob.solve())
print("A solution y is")
print(y.value)

Optimal MAD: 66.6423937007172
A solution y is
[-4.71254253e-10  7.40833541e-01  9.49987224e-02  1.64167738e-01
 -1.45087935e-09  3.40318129e+00  1.35199879e-09  1.72698743e+01
 -1.49707579e-09  3.95851858e+00  8.68962273e+00  2.16088131e-09
 -5.16163273e-10  2.79619896e-10 -1.23534470e-09 -1.59478289e-09
  7.71818926e-01 -2.26033127e-09  1.44109286e+01  8.96512419e-10
  4.69581578e+00 -1.28964857e-09  2.94205355e-10  8.93464927e-11
  2.10146336e+00  8.21835708e+00  2.79661544e+00  3.26197639e-01]


In [83]:
#Allocation weights
allocation = y.value[:4]
allocation

array([-4.71254253e-10,  7.40833541e-01,  9.49987224e-02,  1.64167738e-01])

In [84]:
#Expected portfolio return
mean_vector@allocation

8.999999997161723